# Syndrome Extraction

In the previous steps, we studied the stabilizer structure, logical operators, encoding, and different error models of the rotated distance-3 surface code.

In this step, we will implement **syndrome extraction** using ancilla qubits.

The purpose of syndrome extraction is to determine whether an error has occurred on the data qubits without directly measuring the quantum information stored in the logical qubit.

### Surface Code Used

We use a rotated distance-3 surface code with:

* 9 data qubits
* 4 X-type stabilizers
* 4 Z-type stabilizers
* 1 logical qubit

The code parameters are:

$$
[[9,1,3]]
$$

### Logical Operators
The logical operators used in this implementation are:

$$
\bar{X}=X_1X_2X_3
$$

$$
\bar{Z}=Z_3Z_6Z_9
$$

### Ancilla Qubits
To measure the stabilizers, we introduce ancilla qubits.
Therefore, the syndrome-extraction circuit will contain:

* 9 data qubits
* 4 X-stabilizer ancillas
* 4 Z-stabilizer ancillas

giving a total of

$$
9+4+4=17
$$

qubits.

### Goal
We will construct the syndrome-extraction circuit and verify that different errors produce the expected syndrome measurements.


In [1]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Pauli

In [7]:
from qiskit import QuantumCircuit

# Encoded |0L>
qc = QuantumCircuit(9)

qc.h(0)
qc.h(2)
qc.h(3)
qc.h(4)

qc.cx(0, 1)
qc.cx(0, 5)
qc.cx(0, 6)
qc.cx(0, 7)
qc.cx(0, 8)

qc.cx(2, 5)
qc.cx(3, 6)

qc.cx(4, 5)
qc.cx(4, 7)
qc.cx(4, 8)

print(qc.draw())

     ┌───┐                                                  
q_0: ┤ H ├──■────■────■─────────■──────────────■────────────
     └───┘┌─┴─┐  │    │         │              │            
q_1: ─────┤ X ├──┼────┼─────────┼──────────────┼────────────
     ┌───┐└───┘  │    │         │              │            
q_2: ┤ H ├───────┼────┼────■────┼──────────────┼────────────
     ├───┤       │    │    │    │              │            
q_3: ┤ H ├───────┼────┼────┼────┼────■─────────┼────────────
     ├───┤       │    │    │    │    │         │            
q_4: ┤ H ├───────┼────┼────┼────┼────┼────■────┼────■────■──
     └───┘     ┌─┴─┐  │  ┌─┴─┐  │    │  ┌─┴─┐  │    │    │  
q_5: ──────────┤ X ├──┼──┤ X ├──┼────┼──┤ X ├──┼────┼────┼──
               └───┘┌─┴─┐└───┘  │  ┌─┴─┐└───┘  │    │    │  
q_6: ───────────────┤ X ├───────┼──┤ X ├───────┼────┼────┼──
                    └───┘     ┌─┴─┐└───┘       │  ┌─┴─┐  │  
q_7: ─────────────────────────┤ X ├────────────┼──┤ X ├──┼──
                        

In [2]:
# Encoded |0L>
qc = QuantumCircuit(9, 9)

qc.h(0)
qc.h(2)
qc.h(3)
qc.h(4)

qc.cx(0, 1)
qc.cx(0, 5)
qc.cx(0, 6)
qc.cx(0, 7)
qc.cx(0, 8)

qc.cx(2, 5)
qc.cx(3, 6)

qc.cx(4, 5)
qc.cx(4, 7)
qc.cx(4, 8)

print(qc.draw())

     ┌───┐                                                  
q_0: ┤ H ├──■────■────■─────────■──────────────■────────────
     └───┘┌─┴─┐  │    │         │              │            
q_1: ─────┤ X ├──┼────┼─────────┼──────────────┼────────────
     ┌───┐└───┘  │    │         │              │            
q_2: ┤ H ├───────┼────┼────■────┼──────────────┼────────────
     ├───┤       │    │    │    │              │            
q_3: ┤ H ├───────┼────┼────┼────┼────■─────────┼────────────
     ├───┤       │    │    │    │    │         │            
q_4: ┤ H ├───────┼────┼────┼────┼────┼────■────┼────■────■──
     └───┘     ┌─┴─┐  │  ┌─┴─┐  │    │  ┌─┴─┐  │    │    │  
q_5: ──────────┤ X ├──┼──┤ X ├──┼────┼──┤ X ├──┼────┼────┼──
               └───┘┌─┴─┐└───┘  │  ┌─┴─┐└───┘  │    │    │  
q_6: ───────────────┤ X ├───────┼──┤ X ├───────┼────┼────┼──
                    └───┘     ┌─┴─┐└───┘       │  ┌─┴─┐  │  
q_7: ─────────────────────────┤ X ├────────────┼──┤ X ├──┼──
                        

In [3]:
# Stabilizer definition:
def pauli_from_qubits(qubits, pauli_type, n=9):
    labels = ["I"] * n

    for q in qubits:
        labels[q - 1] = pauli_type

    return Pauli("".join(labels[::-1]))

In [4]:
# X and Z stabilizers:
X_stabilizers = [
    pauli_from_qubits([1, 2, 4, 5], "X"),
    pauli_from_qubits([3, 6], "X"),
    pauli_from_qubits([5, 6, 8, 9], "X"),
    pauli_from_qubits([4, 7], "X")
]

Z_stabilizers = [
    pauli_from_qubits([1, 2], "Z"),
    pauli_from_qubits([2, 3, 5, 6], "Z"),
    pauli_from_qubits([4, 5, 7, 8], "Z"),
    pauli_from_qubits([8, 9], "Z")
]

### Qubit Assignment

For syndrome extraction, we use 17 qubits:

* D1–D9 → data qubits
* X1–X4 → ancilla qubits for measuring X-type stabilizers
* Z1–Z4 → ancilla qubits for measuring Z-type stabilizers

The qubit indices are assigned as follows:

$$
\begin{aligned}
D1-D9 &: 0-8\\
X1-X4 &: 9-12\\
Z1-Z4 &: 13-16
\end{aligned}
$$

Thus, the complete syndrome-extraction circuit contains 17 qubits.

In [5]:
# Create a 17-qubit circuit
qc17 = QuantumCircuit(17, 8)

print(qc17.draw())

      
 q_0: 
      
 q_1: 
      
 q_2: 
      
 q_3: 
      
 q_4: 
      
 q_5: 
      
 q_6: 
      
 q_7: 
      
 q_8: 
      
 q_9: 
      
q_10: 
      
q_11: 
      
q_12: 
      
q_13: 
      
q_14: 
      
q_15: 
      
q_16: 
      
 c: 8/
      


In [9]:
# Create a fresh 17-qubit, 8-classical-bit circuit
qc17 = QuantumCircuit(17, 8)

# Add encoded |0L> state preparation
qc17.compose(
    qc,
    qubits=range(9),
    inplace=True
)

print(qc17.draw(fold=-1))

      ┌───┐                                                  
 q_0: ┤ H ├──■────■────■─────────■──────────────■────────────
      └───┘┌─┴─┐  │    │         │              │            
 q_1: ─────┤ X ├──┼────┼─────────┼──────────────┼────────────
      ┌───┐└───┘  │    │         │              │            
 q_2: ┤ H ├───────┼────┼────■────┼──────────────┼────────────
      ├───┤       │    │    │    │              │            
 q_3: ┤ H ├───────┼────┼────┼────┼────■─────────┼────────────
      ├───┤       │    │    │    │    │         │            
 q_4: ┤ H ├───────┼────┼────┼────┼────┼────■────┼────■────■──
      └───┘     ┌─┴─┐  │  ┌─┴─┐  │    │  ┌─┴─┐  │    │    │  
 q_5: ──────────┤ X ├──┼──┤ X ├──┼────┼──┤ X ├──┼────┼────┼──
                └───┘┌─┴─┐└───┘  │  ┌─┴─┐└───┘  │    │    │  
 q_6: ───────────────┤ X ├───────┼──┤ X ├───────┼────┼────┼──
                     └───┘     ┌─┴─┐└───┘       │  ┌─┴─┐  │  
 q_7: ─────────────────────────┤ X ├────────────┼──┤ X ├──┼──
        

In [10]:
from qiskit import QuantumCircuit

# Fresh 17-qubit, 8-classical-bit circuit
qc17 = QuantumCircuit(17, 8)

# Add encoded |0_L> state preparation
qc17.compose(
    qc,
    qubits=range(9),
    inplace=True
)

# X1 stabilizer measurement
# X1 = D1 D2 D4 D5

# 1. Prepare X1 ancilla in |+>
qc17.h(9)

# 2. Entangle X1 ancilla with data qubits
qc17.cx(9, 0)   # X1 ancilla -> D1
qc17.cx(9, 1)   # X1 ancilla -> D2
qc17.cx(9, 3)   # X1 ancilla -> D4
qc17.cx(9, 4)   # X1 ancilla -> D5

# 3. Rotate back from X basis
qc17.h(9)

# 4. Measure X1 ancilla
qc17.measure(9, 0)

print(qc17.draw(fold=-1))

      ┌───┐                                                  ┌───┐                       
 q_0: ┤ H ├──■────■────■─────────■──────────────■────────────┤ X ├───────────────────────
      └───┘┌─┴─┐  │    │         │              │            └─┬─┘┌───┐                  
 q_1: ─────┤ X ├──┼────┼─────────┼──────────────┼──────────────┼──┤ X ├──────────────────
      ┌───┐└───┘  │    │         │              │              │  └─┬─┘                  
 q_2: ┤ H ├───────┼────┼────■────┼──────────────┼──────────────┼────┼────────────────────
      ├───┤       │    │    │    │              │              │    │  ┌───┐             
 q_3: ┤ H ├───────┼────┼────┼────┼────■─────────┼──────────────┼────┼──┤ X ├─────────────
      ├───┤       │    │    │    │    │         │              │    │  └─┬─┘┌───┐        
 q_4: ┤ H ├───────┼────┼────┼────┼────┼────■────┼────■────■────┼────┼────┼──┤ X ├────────
      └───┘     ┌─┴─┐  │  ┌─┴─┐  │    │  ┌─┴─┐  │    │    │    │    │    │  └─┬─┘        
 q_5: ────

In [11]:
from qiskit_aer import AerSimulator

simulator = AerSimulator()

result = simulator.run(qc17, shots=100).result()

counts = result.get_counts()

print(counts)

{'00000000': 100}


### X1 Stabilizer Measurement

The encoded state \(|0_L\rangle\) is a +1 eigenstate of the X1 stabilizer:

\[
X_1 = X_{D1}X_{D2}X_{D4}X_{D5}
\]

Therefore,

\[
X_1|0_L\rangle = |0_L\rangle
\]

The X1 ancilla was measured 100 times, and the result was:

```text
{'00000000000000000': 100}

In [12]:
# X2 stabilizer measurement
# X2 = D3 D6

qc17.h(10)

qc17.cx(10, 2)   # D3
qc17.cx(10, 5)   # D6

qc17.h(10)

qc17.measure(10, 1)

print(qc17.draw(fold=-1))

      ┌───┐                                                            ┌───┐                       
 q_0: ┤ H ├──■────■────■─────────■───────────────────■─────────────────┤ X ├───────────────────────
      └───┘┌─┴─┐  │    │         │                   │                 └─┬─┘┌───┐                  
 q_1: ─────┤ X ├──┼────┼─────────┼───────────────────┼───────────────────┼──┤ X ├──────────────────
      ┌───┐└───┘  │    │         │            ┌───┐  │                   │  └─┬─┘                  
 q_2: ┤ H ├───────┼────┼────■────┼────────────┤ X ├──┼───────────────────┼────┼────────────────────
      ├───┤       │    │    │    │            └─┬─┘  │                   │    │  ┌───┐             
 q_3: ┤ H ├───────┼────┼────┼────┼────■─────────┼────┼───────────────────┼────┼──┤ X ├─────────────
      ├───┤       │    │    │    │    │         │    │                   │    │  └─┬─┘┌───┐        
 q_4: ┤ H ├───────┼────┼────┼────┼────┼────■────┼────┼────■─────────■────┼────┼────┼──┤ X ├────────


In [13]:
from qiskit_aer import AerSimulator

simulator = AerSimulator()

result = simulator.run(qc17, shots=100).result()

counts = result.get_counts()

print(counts)

{'00000000': 100}


In [14]:
# X3 stabilizer measurement
# X3 = D5 D6 D8 D9

# Prepare X3 ancilla in |+>
qc17.h(11)

# Entangle with the data qubits
qc17.cx(11, 4)   # D5
qc17.cx(11, 5)   # D6
qc17.cx(11, 7)   # D8
qc17.cx(11, 8)   # D9

# Rotate back
qc17.h(11)

# Measure X3 ancilla
qc17.measure(11, 2)

print(qc17.draw(fold=-1))

      ┌───┐                                                            ┌───┐                                                   
 q_0: ┤ H ├──■────■────■─────────■───────────────────■─────────────────┤ X ├───────────────────────────────────────────────────
      └───┘┌─┴─┐  │    │         │                   │                 └─┬─┘┌───┐                                              
 q_1: ─────┤ X ├──┼────┼─────────┼───────────────────┼───────────────────┼──┤ X ├──────────────────────────────────────────────
      ┌───┐└───┘  │    │         │            ┌───┐  │                   │  └─┬─┘                                              
 q_2: ┤ H ├───────┼────┼────■────┼────────────┤ X ├──┼───────────────────┼────┼────────────────────────────────────────────────
      ├───┤       │    │    │    │            └─┬─┘  │                   │    │  ┌───┐                                         
 q_3: ┤ H ├───────┼────┼────┼────┼────■─────────┼────┼───────────────────┼────┼──┤ X ├──────────────────

In [15]:
from qiskit_aer import AerSimulator

simulator = AerSimulator()

result = simulator.run(qc17, shots=100).result()

counts = result.get_counts()

print(counts)

{'00000000': 100}


In [16]:
# X4 stabilizer measurement
# X4 = D4 D7

# Prepare X4 ancilla in |+>
qc17.h(12)

# Entangle with the data qubits
qc17.cx(12, 3)   # D4
qc17.cx(12, 6)   # D7

# Rotate back
qc17.h(12)

# Measure X4 ancilla
qc17.measure(12, 3)

print(qc17.draw(fold=-1))

      ┌───┐                                                            ┌───┐                                                             
 q_0: ┤ H ├──■────■────■─────────■───────────────────■─────────────────┤ X ├─────────────────────────────────────────────────────────────
      └───┘┌─┴─┐  │    │         │                   │                 └─┬─┘┌───┐                                                        
 q_1: ─────┤ X ├──┼────┼─────────┼───────────────────┼───────────────────┼──┤ X ├────────────────────────────────────────────────────────
      ┌───┐└───┘  │    │         │            ┌───┐  │                   │  └─┬─┘                                                        
 q_2: ┤ H ├───────┼────┼────■────┼────────────┤ X ├──┼───────────────────┼────┼──────────────────────────────────────────────────────────
      ├───┤       │    │    │    │            └─┬─┘  │                   │    │  ┌───┐     ┌───┐                                         
 q_3: ┤ H ├───────┼────┼────┼────┼

In [17]:
result = simulator.run(qc17, shots=100).result()
counts = result.get_counts()

print(counts)

{'00000000': 100}


In [18]:
# Z1 stabilizer measurement
# Z1 = D1 D2

# Entangle data qubits with Z1 ancilla
qc17.cx(0, 13)   # D1 -> Z1 ancilla
qc17.cx(1, 13)   # D2 -> Z1 ancilla

# Measure Z1 ancilla
qc17.measure(13, 4)

print(qc17.draw(fold=-1))

      ┌───┐                                                            ┌───┐                                                                       
 q_0: ┤ H ├──■────■────■─────────■───────────────────■─────────────────┤ X ├───────■───────────────────────────────────────────────────────────────
      └───┘┌─┴─┐  │    │         │                   │                 └─┬─┘┌───┐  │                                                               
 q_1: ─────┤ X ├──┼────┼─────────┼───────────────────┼───────────────────┼──┤ X ├──┼─────────■─────────────────────────────────────────────────────
      ┌───┐└───┘  │    │         │            ┌───┐  │                   │  └─┬─┘  │         │                                                     
 q_2: ┤ H ├───────┼────┼────■────┼────────────┤ X ├──┼───────────────────┼────┼────┼─────────┼─────────────────────────────────────────────────────
      ├───┤       │    │    │    │            └─┬─┘  │                   │    │    │  ┌───┐  │       ┌───┐      

In [19]:
result = simulator.run(qc17, shots=100).result()
counts = result.get_counts()

print(counts)

{'00000000': 100}


In [20]:
# Z2 stabilizer measurement
# Z2 = D2 D3 D5 D6

qc17.cx(1, 14)   # D2 -> Z2 ancilla
qc17.cx(2, 14)   # D3 -> Z2 ancilla
qc17.cx(4, 14)   # D5 -> Z2 ancilla
qc17.cx(5, 14)   # D6 -> Z2 ancilla

qc17.measure(14, 5)

print(qc17.draw(fold=-1))

      ┌───┐                                                            ┌───┐                                                                                           
 q_0: ┤ H ├──■────■────■─────────■───────────────────■─────────────────┤ X ├───────■───────────────────────────────────────────────────────────────────────────────────
      └───┘┌─┴─┐  │    │         │                   │                 └─┬─┘┌───┐  │                                                                                   
 q_1: ─────┤ X ├──┼────┼─────────┼───────────────────┼───────────────────┼──┤ X ├──┼─────────■──────────────■──────────────────────────────────────────────────────────
      ┌───┐└───┘  │    │         │            ┌───┐  │                   │  └─┬─┘  │         │              │                                                          
 q_2: ┤ H ├───────┼────┼────■────┼────────────┤ X ├──┼───────────────────┼────┼────┼─────────┼──────────────┼───────────────────■───────────────────────────────

In [21]:
result = simulator.run(qc17, shots=100).result()
counts = result.get_counts()

print(counts)

{'00000000': 100}


In [22]:
# Z3 stabilizer measurement
# Z3 = D4 D5 D7 D8

qc17.cx(3, 15)   # D4 -> Z3 ancilla
qc17.cx(4, 15)   # D5 -> Z3 ancilla
qc17.cx(6, 15)   # D7 -> Z3 ancilla
qc17.cx(7, 15)   # D8 -> Z3 ancilla

qc17.measure(15, 6)

print(qc17.draw(fold=-1))

      ┌───┐                                                            ┌───┐                                                                                                                  
 q_0: ┤ H ├──■────■────■─────────■───────────────────■─────────────────┤ X ├───────■──────────────────────────────────────────────────────────────────────────────────────────────────────────
      └───┘┌─┴─┐  │    │         │                   │                 └─┬─┘┌───┐  │                                                                                                          
 q_1: ─────┤ X ├──┼────┼─────────┼───────────────────┼───────────────────┼──┤ X ├──┼─────────■──────────────■─────────────────────────────────────────────────────────────────────────────────
      ┌───┐└───┘  │    │         │            ┌───┐  │                   │  └─┬─┘  │         │              │                                                                                 
 q_2: ┤ H ├───────┼────┼────■────┼───────────

In [23]:
# Z4 stabilizer measurement
# Z4 = D8 D9

qc17.cx(7, 16)   # D8 -> Z4 ancilla
qc17.cx(8, 16)   # D9 -> Z4 ancilla

qc17.measure(16, 7)

print(qc17.draw(fold=-1))

      ┌───┐                                                            ┌───┐                                                                                                                               
 q_0: ┤ H ├──■────■────■─────────■───────────────────■─────────────────┤ X ├───────■───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      └───┘┌─┴─┐  │    │         │                   │                 └─┬─┘┌───┐  │                                                                                                                       
 q_1: ─────┤ X ├──┼────┼─────────┼───────────────────┼───────────────────┼──┤ X ├──┼─────────■──────────────■──────────────────────────────────────────────────────────────────────────────────────────────
      ┌───┐└───┘  │    │         │            ┌───┐  │                   │  └─┬─┘  │         │              │                                                                           

In [24]:
result = simulator.run(qc17, shots=100).result()
counts = result.get_counts()

print(counts)

{'00000000': 100}


### Syndrome Extraction: Error-Free Encoded State

The encoded \(|0_L\rangle\) state was prepared and all eight stabilizers were measured using ancilla qubits.

The measured syndrome was:

\[
S_X = [0,0,0,0]
\]

\[
S_Z = [0,0,0,0]
\]

Therefore, the complete syndrome is:

\[
S = [0,0,0,0,0,0,0,0]
\]

Since no error was introduced, all syndrome bits are zero, confirming that the encoded state satisfies all eight stabilizer checks.

In [26]:
# Fresh circuit for testing an X error on D5
qc_error = QuantumCircuit(17, 8)

# Prepare encoded |0_L>
qc_error.compose(
    qc,
    qubits=range(9),
    inplace=True
)

# Introduce X error on D5
qc_error.x(4)

print(qc_error.draw(fold=-1))

      ┌───┐                                                       
 q_0: ┤ H ├──■────■────■─────────■──────────────■─────────────────
      └───┘┌─┴─┐  │    │         │              │                 
 q_1: ─────┤ X ├──┼────┼─────────┼──────────────┼─────────────────
      ┌───┐└───┘  │    │         │              │                 
 q_2: ┤ H ├───────┼────┼────■────┼──────────────┼─────────────────
      ├───┤       │    │    │    │              │                 
 q_3: ┤ H ├───────┼────┼────┼────┼────■─────────┼─────────────────
      ├───┤       │    │    │    │    │         │            ┌───┐
 q_4: ┤ H ├───────┼────┼────┼────┼────┼────■────┼────■────■──┤ X ├
      └───┘     ┌─┴─┐  │  ┌─┴─┐  │    │  ┌─┴─┐  │    │    │  └───┘
 q_5: ──────────┤ X ├──┼──┤ X ├──┼────┼──┤ X ├──┼────┼────┼───────
                └───┘┌─┴─┐└───┘  │  ┌─┴─┐└───┘  │    │    │       
 q_6: ───────────────┤ X ├───────┼──┤ X ├───────┼────┼────┼───────
                     └───┘     ┌─┴─┐└───┘       │  ┌─┴─┐  │   

In [27]:
# Z1 = D1 D2
qc_error.cx(0, 13)   # D1 -> Z1 ancilla
qc_error.cx(1, 13)   # D2 -> Z1 ancilla
qc_error.measure(13, 4)

In [28]:
# Z2 = D2 D3 D5 D6
qc_error.cx(1, 14)   # D2
qc_error.cx(2, 14)   # D3
qc_error.cx(4, 14)   # D5
qc_error.cx(5, 14)   # D6
qc_error.measure(14, 5)

In [29]:
# Z3 = D4 D5 D7 D8
qc_error.cx(3, 15)   # D4
qc_error.cx(4, 15)   # D5
qc_error.cx(6, 15)   # D7
qc_error.cx(7, 15)   # D8
qc_error.measure(15, 6)

In [30]:
# Z4 = D8 D9
qc_error.cx(7, 16)   # D8
qc_error.cx(8, 16)   # D9
qc_error.measure(16, 7)

In [31]:
print(qc_error.draw(fold=-1))

      ┌───┐                                                                                                                               
 q_0: ┤ H ├──■────■────■─────────■──────────────■───────────────────■─────────────────────────────────────────────────────────────────────
      └───┘┌─┴─┐  │    │         │              │                   │                                                                     
 q_1: ─────┤ X ├──┼────┼─────────┼──────────────┼───────────────────┼─────────■───────■───────────────────────────────────────────────────
      ┌───┐└───┘  │    │         │              │                   │         │       │                                                   
 q_2: ┤ H ├───────┼────┼────■────┼──────────────┼───────────────────┼─────────┼───────┼────■──────────────────────────────────────────────
      ├───┤       │    │    │    │              │                   │         │       │    │                                              
 q_3: ┤ H ├───────┼────┼───

In [32]:
result = simulator.run(qc_error, shots=100).result()
counts = result.get_counts()

print(counts)

{'01100000': 100}


In [33]:
# Z stabilizers
Z_stabilizers_qubits = [
    [1, 2],        # Z1
    [2, 3, 5, 6],  # Z2
    [4, 5, 7, 8],  # Z3
    [8, 9]         # Z4
]

print("X error -> Z syndrome")

for d in range(1, 10):
    syndrome = []

    for stab in Z_stabilizers_qubits:
        if d in stab:
            syndrome.append(1)
        else:
            syndrome.append(0)

    print(f"X{d} -> {syndrome}")

X error -> Z syndrome
X1 -> [1, 0, 0, 0]
X2 -> [1, 1, 0, 0]
X3 -> [0, 1, 0, 0]
X4 -> [0, 0, 1, 0]
X5 -> [0, 1, 1, 0]
X6 -> [0, 1, 0, 0]
X7 -> [0, 0, 1, 0]
X8 -> [0, 0, 1, 1]
X9 -> [0, 0, 0, 1]


In [34]:
def test_X_error(error_qubit):

    qc_test = QuantumCircuit(17, 8)

    # Encode |0_L>
    qc_test.compose(qc, qubits=range(9), inplace=True)

    # X error
    qc_test.x(error_qubit)

    # Z1
    qc_test.cx(0, 13)
    qc_test.cx(1, 13)
    qc_test.measure(13, 4)

    # Z2
    qc_test.cx(1, 14)
    qc_test.cx(2, 14)
    qc_test.cx(4, 14)
    qc_test.cx(5, 14)
    qc_test.measure(14, 5)

    # Z3
    qc_test.cx(3, 15)
    qc_test.cx(4, 15)
    qc_test.cx(6, 15)
    qc_test.cx(7, 15)
    qc_test.measure(15, 6)

    # Z4
    qc_test.cx(7, 16)
    qc_test.cx(8, 16)
    qc_test.measure(16, 7)

    return qc_test

In [35]:
for q in range(9):

    circuit = test_X_error(q)

    result = simulator.run(circuit, shots=100).result()
    counts = result.get_counts()

    print(f"D{q+1} X error:", counts)

D1 X error: {'00010000': 100}
D2 X error: {'00110000': 100}
D3 X error: {'00100000': 100}
D4 X error: {'01000000': 100}
D5 X error: {'01100000': 100}
D6 X error: {'00100000': 100}
D7 X error: {'01000000': 100}
D8 X error: {'11000000': 100}
D9 X error: {'10000000': 100}


### Single-Qubit X Error Syndrome Extraction

To verify the syndrome extraction circuit, a single-qubit \(X\) error was introduced independently on each of the 9 data qubits \(D1-D9\).

Since an \(X\) error anticommutes with \(Z\)-type stabilizers, the \(Z\)-stabilizer measurements were used to detect the error.

The measured syndromes were:

```text
D1 X error: {'00010000': 100}
D2 X error: {'00110000': 100}
D3 X error: {'00100000': 100}
D4 X error: {'01000000': 100}
D5 X error: {'01100000': 100}
D6 X error: {'00100000': 100}
D7 X error: {'01000000': 100}
D8 X error: {'11000000': 100}
D9 X error: {'10000000': 100}
```

The classical bits are displayed by Qiskit in reverse order. Therefore, the measured bit strings are interpreted according to the stabilizer ordering

$$
[X_1,X_2,X_3,X_4 \mid Z_1,Z_2,Z_3,Z_4].
$$

For example, for an \(X\) error on \(D5\), Qiskit returns:

```text
01100000
```

which corresponds to the logical syndrome:

```text
00000110
```

Thus, \(Z_2\) and \(Z_3\) are flipped.

This confirms that the syndrome extraction circuit correctly detects single-qubit \(X\) errors on all nine data qubits.


In [36]:
def test_Z_error(error_qubit):

    qc_test = QuantumCircuit(17, 8)

    # Encode |0_L>
    qc_test.compose(qc, qubits=range(9), inplace=True)

    # Z error
    qc_test.z(error_qubit)

    # X1
    qc_test.h(9)
    qc_test.cx(9, 0)
    qc_test.cx(9, 1)
    qc_test.cx(9, 3)
    qc_test.cx(9, 4)
    qc_test.h(9)
    qc_test.measure(9, 0)

    # X2
    qc_test.h(10)
    qc_test.cx(10, 2)
    qc_test.cx(10, 5)
    qc_test.h(10)
    qc_test.measure(10, 1)

    # X3
    qc_test.h(11)
    qc_test.cx(11, 4)
    qc_test.cx(11, 5)
    qc_test.cx(11, 7)
    qc_test.cx(11, 8)
    qc_test.h(11)
    qc_test.measure(11, 2)

    # X4
    qc_test.h(12)
    qc_test.cx(12, 3)
    qc_test.cx(12, 6)
    qc_test.h(12)
    qc_test.measure(12, 3)

    return qc_test

In [37]:
for q in range(9):

    circuit = test_Z_error(q)

    result = simulator.run(circuit, shots=100).result()
    counts = result.get_counts()

    print(f"D{q+1} Z error:", counts)

D1 Z error: {'00000001': 100}
D2 Z error: {'00000001': 100}
D3 Z error: {'00000010': 100}
D4 Z error: {'00001001': 100}
D5 Z error: {'00000101': 100}
D6 Z error: {'00000110': 100}
D7 Z error: {'00001000': 100}
D8 Z error: {'00000100': 100}
D9 Z error: {'00000100': 100}


In [39]:
qc_test = QuantumCircuit(17, 8)

# Encode |0_L>
qc_test.compose(qc, qubits=range(9), inplace=True)

# Z error on D3
qc_test.z(2)   # D3 = qubit 2

# X1 measurement
qc_test.h(9)
qc_test.cx(9, 0)
qc_test.cx(9, 1)
qc_test.cx(9, 3)
qc_test.cx(9, 4)
qc_test.h(9)
qc_test.measure(9, 0)

# X2 measurement
qc_test.h(10)
qc_test.cx(10, 2)
qc_test.cx(10, 5)
qc_test.h(10)
qc_test.measure(10, 1)

# X3 measurement
qc_test.h(11)
qc_test.cx(11, 4)
qc_test.cx(11, 5)
qc_test.cx(11, 7)
qc_test.cx(11, 8)
qc_test.h(11)
qc_test.measure(11, 2)

# X4 measurement
qc_test.h(12)
qc_test.cx(12, 3)
qc_test.cx(12, 6)
qc_test.h(12)
qc_test.measure(12, 3)

# Run
result = simulator.run(qc_test, shots=100).result()
counts = result.get_counts()

print(counts)

{'00000010': 100}


### Single-Qubit Z Error Syndrome Extraction

Next, single-qubit \(Z\) errors were introduced independently on each of the 9 data qubits \(D1-D9\).

Since a \(Z\) error anticommutes with \(X\)-type stabilizers, the \(X\)-stabilizer measurements were used to detect the error.

The measured Qiskit outputs were:

```text
D1 Z error: {'00000001': 100}
D2 Z error: {'00000001': 100}
D3 Z error: {'00000010': 100}
D4 Z error: {'00001001': 100}
D5 Z error: {'00000101': 100}
D6 Z error: {'00000110': 100}
D7 Z error: {'00001000': 100}
D8 Z error: {'00000100': 100}
D9 Z error: {'00000100': 100}
```

The classical bit strings are displayed by Qiskit in reverse order. Therefore, the strings are reversed to obtain the logical syndrome ordering

$$
[X_1,X_2,X_3,X_4 \mid Z_1,Z_2,Z_3,Z_4].
$$

For example, for a \(Z\) error on \(D3\), the simulation gives:

```text
00000010
```

Reversing the bit string:

```text
00000010 → 01000000
```

Thus, the logical syndrome is

```text
0100 | 0000
```

Only \(X_2\) is flipped.

This is consistent with

$$
X_2 = X_{D3}X_{D6}.
$$

Since \(D3\) is part of \(X_2\), a \(Z\) error on \(D3\) anticommutes with \(X_2\), giving

$$
\boxed{Z(D3)\rightarrow X_2}.
$$

The results confirm that the syndrome extraction circuit correctly detects single-qubit \(Z\) errors on all nine data qubits.


In [40]:
qc_test = QuantumCircuit(17, 8)

# Encode |0_L>
qc_test.compose(qc, qubits=range(9), inplace=True)

# Y error on D5
qc_test.y(4)   # D5 = qubit 4

In [41]:
# X1
qc_test.h(9)
qc_test.cx(9, 0)
qc_test.cx(9, 1)
qc_test.cx(9, 3)
qc_test.cx(9, 4)
qc_test.h(9)
qc_test.measure(9, 0)

# X2
qc_test.h(10)
qc_test.cx(10, 2)
qc_test.cx(10, 5)
qc_test.h(10)
qc_test.measure(10, 1)

# X3
qc_test.h(11)
qc_test.cx(11, 4)
qc_test.cx(11, 5)
qc_test.cx(11, 7)
qc_test.cx(11, 8)
qc_test.h(11)
qc_test.measure(11, 2)

# X4
qc_test.h(12)
qc_test.cx(12, 3)
qc_test.cx(12, 6)
qc_test.h(12)
qc_test.measure(12, 3)

In [42]:
# Z1
qc_test.cx(0, 13)
qc_test.cx(1, 13)
qc_test.measure(13, 4)

# Z2
qc_test.cx(1, 14)
qc_test.cx(2, 14)
qc_test.cx(4, 14)
qc_test.cx(5, 14)
qc_test.measure(14, 5)

# Z3
qc_test.cx(3, 15)
qc_test.cx(4, 15)
qc_test.cx(6, 15)
qc_test.cx(7, 15)
qc_test.measure(15, 6)

# Z4
qc_test.cx(7, 16)
qc_test.cx(8, 16)
qc_test.measure(16, 7)

In [43]:
result = simulator.run(qc_test, shots=100).result()
counts = result.get_counts()

print(counts)

{'01100101': 100}


In [44]:
def test_Y_error(error_qubit):

    qc_test = QuantumCircuit(17, 8)

    # Encode |0_L>
    qc_test.compose(qc, qubits=range(9), inplace=True)

    # Y error
    qc_test.y(error_qubit)

    # -------- X stabilizers --------

    # X1
    qc_test.h(9)
    qc_test.cx(9, 0)
    qc_test.cx(9, 1)
    qc_test.cx(9, 3)
    qc_test.cx(9, 4)
    qc_test.h(9)
    qc_test.measure(9, 0)

    # X2
    qc_test.h(10)
    qc_test.cx(10, 2)
    qc_test.cx(10, 5)
    qc_test.h(10)
    qc_test.measure(10, 1)

    # X3
    qc_test.h(11)
    qc_test.cx(11, 4)
    qc_test.cx(11, 5)
    qc_test.cx(11, 7)
    qc_test.cx(11, 8)
    qc_test.h(11)
    qc_test.measure(11, 2)

    # X4
    qc_test.h(12)
    qc_test.cx(12, 3)
    qc_test.cx(12, 6)
    qc_test.h(12)
    qc_test.measure(12, 3)

    # -------- Z stabilizers --------

    # Z1
    qc_test.cx(0, 13)
    qc_test.cx(1, 13)
    qc_test.measure(13, 4)

    # Z2
    qc_test.cx(1, 14)
    qc_test.cx(2, 14)
    qc_test.cx(4, 14)
    qc_test.cx(5, 14)
    qc_test.measure(14, 5)

    # Z3
    qc_test.cx(3, 15)
    qc_test.cx(4, 15)
    qc_test.cx(6, 15)
    qc_test.cx(7, 15)
    qc_test.measure(15, 6)

    # Z4
    qc_test.cx(7, 16)
    qc_test.cx(8, 16)
    qc_test.measure(16, 7)

    return qc_test

In [45]:
for q in range(9):

    circuit = test_Y_error(q)

    result = simulator.run(circuit, shots=100).result()
    counts = result.get_counts()

    print(f"D{q+1} Y error:", counts)

D1 Y error: {'00010001': 100}
D2 Y error: {'00110001': 100}
D3 Y error: {'00100010': 100}
D4 Y error: {'01001001': 100}
D5 Y error: {'01100101': 100}
D6 Y error: {'00100110': 100}
D7 Y error: {'01001000': 100}
D8 Y error: {'11000100': 100}
D9 Y error: {'10000100': 100}


## Degeneracy in Quantum Error Correction

In a stabilizer code, two different physical errors can sometimes produce the same syndrome.

Therefore, a syndrome does not always uniquely identify the physical error.

However, **having the same syndrome does not automatically mean that two errors are degenerate**.

For two errors \(E_1\) and \(E_2\),

$$
S(E_1)=S(E_2)
$$

means that \(E_1E_2\) commutes with all stabilizers.

To determine whether the two errors are truly degenerate, we examine their product \(E_1E_2\).

### Case 1: Degenerate Errors

If

$$
E_1E_2 \in \mathcal{S},
$$

where \(\mathcal{S}\) is the stabilizer group, then \(E_1\) and \(E_2\) are **degenerate**.

In this case, the two errors have the same effect on the encoded logical state and cannot be distinguished by the code.

### Case 2: Same Syndrome but Not Degenerate

If

$$
E_1E_2
$$

commutes with all stabilizers but is not an element of the stabilizer group, then the two errors have the same syndrome but are **not degenerate**.

In this case, \(E_1E_2\) belongs to the normalizer of the stabilizer group and may contain a non-trivial logical operator.

Therefore,

$$
\boxed{\text{Same syndrome} \neq \text{necessarily degenerate}}
$$

The correct criterion for degeneracy is

$$
\boxed{E_1E_2 \in \mathcal{S}}
$$

where \(\mathcal{S}\) is the stabilizer group.


### Example: Degenerate Errors in the Surface Code

From the measured syndrome table,

$$
X(D3)\rightarrow0100
$$

and

$$
X(D6)\rightarrow0100.
$$

Thus, \(X(D3)\) and \(X(D6)\) produce the same syndrome.

To determine whether they are degenerate, we calculate their product:

$$
E_1E_2=X(D3)X(D6).
$$

Using the stabilizer definition,

$$
X_2=X_{D3}X_{D6},
$$

we obtain

$$
E_1E_2=X_2.
$$

Since \(X_2\) is a member of the stabilizer group,

$$
X(D3)X(D6)\in\mathcal{S}.
$$

Therefore,

$$
\boxed{X(D3)\text{ and }X(D6)\text{ are degenerate errors}.}
$$

They produce the same syndrome and have the same effect on the encoded logical state.


In [ ]:
# Same syndrome check

# X-error syndrome table
X_error_syndromes = {
    "D1": "1000",
    "D2": "1100",
    "D3": "0100",
    "D4": "0010",
    "D5": "0110",
    "D6": "0100",
    "D7": "0010",
    "D8": "0011",
    "D9": "0001"
}

print("X(D3) syndrome =", X_error_syndromes["D3"])
print("X(D6) syndrome =", X_error_syndromes["D6"])

if X_error_syndromes["D3"] == X_error_syndromes["D6"]:
    print("Same syndrome")

X(D3) syndrome = 0100
X(D6) syndrome = 0100
Same syndrome


In [ ]:
# Product check — D3 × D6
X_stabilizer_qubits = {
    "X1": [1, 2, 4, 5],
    "X2": [3, 6],
    "X3": [5, 6, 8, 9],
    "X4": [4, 7]
}

E1 = {3}
E2 = {6}

product = E1.symmetric_difference(E2)

print("E1 =", E1)
print("E2 =", E2)
print("E1 * E2 =", product)

if product == set(X_stabilizer_qubits["X2"]):
    print("E1 * E2 = X2")
    print("Therefore, the two errors are degenerate.")

E1 = {3}
E2 = {6}
E1 * E2 = {3, 6}
E1 * E2 = X2
Therefore, the two errors are degenerate.


In [48]:
syndrome_D1 = "1000"

syndrome_D2 = "1100"
syndrome_D3 = "0100"

syndrome_D2D3 = "1100"  # XOR

In [49]:
def xor_syndrome(s1, s2):
    return "".join(str(int(a) ^ int(b)) for a, b in zip(s1, s2))


syndrome_D2D3 = xor_syndrome(syndrome_D2, syndrome_D3)

print("X(D1) syndrome     =", syndrome_D1)
print("X(D2)X(D3) syndrome =", syndrome_D2D3)

if syndrome_D1 == syndrome_D2D3:
    print("Same syndrome")

X(D1) syndrome     = 1000
X(D2)X(D3) syndrome = 1000
Same syndrome


In [ ]:
# Stabilizer group generate
from qiskit.quantum_info import Pauli


def stabilizer_group(generators):

    group = {Pauli("I" * 9)}

    for g in generators:
        new_elements = set(group)

        for S in group:
            new_elements.add(S.compose(g))

        group = new_elements

    return group


X_group = stabilizer_group(X_stabilizers)
Z_group = stabilizer_group(Z_stabilizers)

print("Number of X stabilizer elements:", len(X_group))
print("Number of Z stabilizer elements:", len(Z_group))

Number of X stabilizer elements: 16
Number of Z stabilizer elements: 16


In [ ]:
# D3/D6 degenerate verify
X3 = pauli_from_qubits([3], "X")
X6 = pauli_from_qubits([6], "X")

product = X3.compose(X6)

print("X(D3) × X(D6) =", product)
print("Degenerate?", product in X_group)

X(D3) × X(D6) = IIIXIIXII
Degenerate? True


In [52]:
Z1 = pauli_from_qubits([1], "Z")
Z2 = pauli_from_qubits([2], "Z")

product = Z1.compose(Z2)

print("Z(D1) × Z(D2) =", product)
print("Degenerate?", product in Z_group)

Z(D1) × Z(D2) = IIIIIIIZZ
Degenerate? True


In [53]:
Z8 = pauli_from_qubits([8], "Z")
Z9 = pauli_from_qubits([9], "Z")

product = Z8.compose(Z9)

print("Z(D8) × Z(D9) =", product)
print("Degenerate?", product in Z_group)

Z(D8) × Z(D9) = ZZIIIIIII
Degenerate? True


In [ ]:
# Same syndrome but not degenerate example — D1 vs D2D3

In [54]:
X_syndrome = {
    "D1": "1000",
    "D2": "1100",
    "D3": "0100",
    "D4": "0010",
    "D5": "0110",
    "D6": "0100",
    "D7": "0010",
    "D8": "0011",
    "D9": "0001"
}

In [55]:
def xor_syndrome(s1, s2):
    return "".join(str(int(a) ^ int(b)) for a, b in zip(s1, s2))


E1_syndrome = X_syndrome["D1"]

E2_syndrome = xor_syndrome(
    X_syndrome["D2"],
    X_syndrome["D3"]
)

print("X(D1) syndrome       =", E1_syndrome)
print("X(D2)X(D3) syndrome  =", E2_syndrome)

X(D1) syndrome       = 1000
X(D2)X(D3) syndrome  = 1000


In [56]:
X1 = pauli_from_qubits([1], "X")
X2 = pauli_from_qubits([2], "X")
X3 = pauli_from_qubits([3], "X")

E1 = X1
E2 = X2.compose(X3)

product = E1.compose(E2)

print("E1 × E2 =", product)

E1 × E2 = IIIIIIXXX


In [57]:
print("Is E1 × E2 a stabilizer?", product in X_group)

Is E1 × E2 a stabilizer? False


In [59]:
X_L = pauli_from_qubits([1, 2, 3], "X")

In [60]:
# Verification, Product = logical X check
X_L = pauli_from_qubits([1, 2, 3], "X")

print("E1 × E2 =", product)
print("X_L     =", X_L)

print("Is product equal to X_L?", product == X_L)

E1 × E2 = IIIIIIXXX
X_L     = IIIIIIXXX
Is product equal to X_L? True


## Why Decoding is Required

Syndrome extraction tells us which stabilizers have been flipped by an error. However, the syndrome does not always uniquely identify the physical error.

For example,

$$
X(D3) \rightarrow 0100
$$

and

$$
X(D6) \rightarrow 0100
$$

produce the same syndrome.

Therefore, after measuring the syndrome, we need a **decoder** to determine an appropriate correction operator.

The decoder takes the measured syndrome as input and finds a correction that is consistent with the observed syndrome while minimizing the probability of a logical error.

The overall process is:

$$
\text{Physical Error}
\rightarrow
\text{Syndrome}
\rightarrow
\text{Decoder}
\rightarrow
\text{Correction}
$$

For the surface code, we will use a **Minimum-Weight Perfect Matching (MWPM)** decoder to determine the most likely correction from the measured syndrome.
